In [ ]:
# Image retrieval task performed with DeepGlobe18 dataset
# TO DO: update documentation

In [ ]:
from google.colab import drive
import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torchvision
from PIL import Image
from sklearn.neighbors import NearestNeighbors
import os
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torch import optim
from tqdm import tqdm
from torchsummary import summary
from google.colab.patches import cv2_imshow
from numpy.linalg import norm
from scipy.spatial import distance

drive.mount('/content/drive')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_dataset_path = '/[...]/train_images'
test_dataset_path = '/[...]/test_images'

In [ ]:
class SatelliteImageryDataset(Dataset):

    def __init__(self, main_dir, transform=None):

        self.main_dir = main_dir
        self.transform = transform
        self.all_imgs = [ f for f in os.listdir(main_dir) if f.endswith('.jpg')]

    def __len__(self):

        return len(self.all_imgs)

    def __getitem__(self, idx):

        img_loc = os.path.join(self.main_dir, self.all_imgs[idx])
        image = Image.open(img_loc).convert("RGB")

        if self.transform is not None:
            tensor_image = self.transform(image)

        return tensor_image, tensor_image


transforms = T.Compose([T.Resize((512, 512)),T.ToTensor()])

tr_dataset=SatelliteImageryDataset(train_dataset_path,
                                   transforms)
print(f'dataset: {len(tr_dataset)} images')
# print(f'dataset[0]: {dataset[0]}')
plt.figure(figsize = (5,5))
plt.imshow(tr_dataset[23][1].permute(1, 2, 0))
tr_dloader = torch.utils.data.DataLoader(tr_dataset, batch_size=64)

class ConvEncoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 16, 3, stride = 2, padding = 1)
        self.selu1 = nn.SELU()

        self.conv2 = nn.Conv2d(16, 32, 5, stride = 2, padding = 1)
        self.selu2 = nn.SELU()

        self.conv3 = nn.Conv2d(32, 64, 7, stride = 5)

    def forward(self, x):

        x = self.conv1(x)
        x = self.selu1(x)

        x = self.conv2(x)
        x = self.selu2(x)

        x = self.conv3(x)

        return x


class ConvDecoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.deconv1 = nn.ConvTranspose2d(64, 32, 7, stride = 5)
        self.selu1 = nn.SELU()

        self.deconv2 = nn.ConvTranspose2d(32, 16, 5, stride=2, padding=1,
                                          output_padding=1)
        self.selu2 = nn.SELU()

        self.deconv3 = nn.ConvTranspose2d(16, 3, 3, stride=2, padding=1,
                                          output_padding=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        x = self.deconv1(x)
        x = self.selu1(x)

        x = self.deconv2(x)
        x = self.selu2(x)

        x = self.deconv3(x)
        x = self.sigmoid(x)

        return x

NUM_EPOCHS=20
LEARNING_RATE=0.001

loss_fn = nn.MSELoss()

encoder = ConvEncoder()
decoder = ConvDecoder()

encoder.to(device)
decoder.to(device)

autoenc_params = list(encoder.parameters()) + list(decoder.parameters())
optimizer = optim.Adam(autoenc_params, lr=LEARNING_RATE)


def train_step(encoder, decoder, train_loader, loss_fn, optimizer, device):

    encoder.train()
    decoder.train()

    for batch_idx, (tr_img, tg_img) in enumerate(train_loader):

        tr_img = tr_img.to(device)
        tg_img = tg_img.to(device)

        optimizer.zero_grad()

        enc_output = encoder(tr_img)
        dec_output = decoder(enc_output)

        loss = loss_fn(dec_output, tg_img)

        loss.backward()
        optimizer.step()

    return loss.item()

In [ ]:
target_image = Image.open(
    test_dataset_path+'/215525_sat.jpg')
transform_to_ten = T.Compose([T.Resize((512, 512)),T.ToTensor()])
image_tensor = transform_to_ten(target_image)
plt.figure(figsize = (3,3))
plt.imshow(image_tensor.permute(1, 2, 0))

In [ ]:
for epoch in tqdm(range(NUM_EPOCHS)):

        train_loss = train_step(encoder, decoder, tr_dloader, loss_fn,
                                optimizer, device=device)
        print(f"Epochs = {epoch}, Training Loss : {train_loss}")

In [ ]:
EMB_CHANNELS=64
EMB_H=25
EMB_W=25
EMB_B=0

def create_embedding(encoder, train_loader, embedding_dim, device):

    encoder.eval()

    embedding = torch.randn(embedding_dim)

    with torch.no_grad():
        for batch_idx, (train_img, target_img) in enumerate(train_loader):

            train_img = train_img.to(device)
            enc_output = encoder(train_img).cpu()

            embedding = torch.cat((embedding, enc_output), 0)

    return embedding


embedding_shape = (EMB_B, EMB_CHANNELS, EMB_H, EMB_W)

embeddings = create_embedding(encoder, tr_dloader, embedding_shape, device)

In [ ]:
NUM_BINS = 256
NUM_IMAGES=50

def get_vector(image, bins=NUM_BINS):

    red = cv.calcHist(

        [image], [2], None, [bins], [0, 256]
    )

    green = cv.calcHist(

        [image], [1], None, [bins], [0, 256]
    )

    blue = cv.calcHist(

        [image], [0], None, [bins], [0, 256]
    )

    vector = np.concatenate([red, green, blue], axis=0)

    vector = vector.reshape(-1)

    return vector



def get_similar_image_indexes(image_tensor, num_images, embeddings, device,
                          encoder):

    image_tensor = image_tensor.unsqueeze(0)

    image_tensor = image_tensor.type(torch.cuda.FloatTensor)

    with torch.no_grad():
        image_embedding = encoder(image_tensor).cpu().detach().numpy()

    flattened_embedding = image_embedding.reshape((image_embedding.shape[0],
                                                   -1))

    knn = NearestNeighbors(n_neighbors=num_images, metric="cosine")
    knn.fit(embeddings.reshape(embeddings.shape[0], -1))

    distances, indices = knn.kneighbors(flattened_embedding)

    indices_list = indices.tolist()
    distances_list = distances.tolist()

    return indices_list[0], distances[0]



def calc_hist_similarity(tr_dataset, indices_list, target_hist):

    dtype = [('image_id', int), ('similarity', float)]
    similarities = []

    for index in indices_list:

            numpy_img = tr_dataset[index][0].numpy().transpose(1, 2, 0) * 255.0
            cv2_img = cv.cvtColor(numpy_img, cv.COLOR_RGB2BGR)
            img_hist = get_vector(cv2_img)

            distance = cv.compareHist(target_hist, img_hist, cv.HISTCMP_CHISQR)

            similarities.append((index, distance))

    hist_similarities = np.array(similarities, dtype=dtype)

    oredered_h_sim = np.sort(hist_similarities, order='similarity')


    return oredered_h_sim



def plot_similar_images(tr_dataset, indices_list):

    fig, axr = plt.subplots(4,4, figsize=(15, 15))
    axr = axr.flatten()

    i = 0

    for (index, value), ax in zip(indices_list, axr):

          if i < 20:

            ax.imshow(tr_dataset[index][0].permute(1, 2, 0))
            i = i+1

          else:
            break


transform_to_ten = T.Compose([T.Resize((512, 512)),T.ToTensor()])

In [ ]:
target_image = Image.open(
    '/[...]/test_images/215525_sat.jpg')
image_tensor = transform_to_ten(target_image)
plt.figure(figsize = (3,3))
plt.imshow(image_tensor.permute(1, 2, 0))

numpy_target_image = image_tensor.numpy().transpose(1, 2, 0) * 255.0
cv2_img_tg = cv.cvtColor(numpy_target_image, cv.COLOR_RGB2BGR)
hist_target_image = get_vector(cv2_img_tg)

similar_idx, similar_dist = get_similar_image_indexes(image_tensor, NUM_IMAGES, embeddings,
                                                  device, encoder)
indices = similar_idx
dist = similar_dist

ordered_histogram_similarities = calc_hist_similarity(tr_dataset, indices,
                                                      hist_target_image)

plot_similar_images(tr_dataset, ordered_histogram_similarities)

In [ ]:
# some example images

transform_ten = T.Compose([T.Resize((512, 512)),T.ToTensor()])
image1 = Image.open('/[...]/test_images/10233_sat.jpg')
image_tensor1 = transform_ten(image1)
image2 = Image.open('/[...]/test_images/159177_sat.jpg')
image_tensor2 = transform_ten(image2)
image3 = Image.open('/[...]/test_images/181447_sat.jpg')
image_tensor3 = transform_ten(image3)
image4 = Image.open('/[...]/test_images/185522_sat.jpg')
image_tensor4 = transform_ten(image4)
image5 = Image.open('/[...]/test_images/147545_sat.jpg')
image_tensor5 = transform_ten(image5)
image6 = Image.open('/[...]/test_images/172307_sat.jpg')
image_tensor6 = transform_ten(image6)
image7 = Image.open('/[...]/test_images/219555_sat.jpg')
image_tensor7 = transform_ten(image7)

In [ ]:
# test image retrieval with one of the images

plt.figure(figsize = (3,3))
plt.imshow(image_tensor1.permute(1, 2, 0))

numpy_image_1 = image_tensor1.numpy().transpose(1, 2, 0) * 255.0
cv2_img_1 = cv.cvtColor(numpy_image_1, cv.COLOR_RGB2BGR)
hist_target_image_1 = get_vector(cv2_img_1)

similar_idx, similar_dist = get_similar_image_indexes(image_tensor1, NUM_IMAGES,
                                                      embeddings,
                                                  device, encoder)
indices = similar_idx

ordered_histogram_similarities = calc_hist_similarity(tr_dataset, indices,
                                                      hist_target_image_1)

plot_similar_images(tr_dataset, ordered_histogram_similarities)